In [ ]:
import pandas as pd
import numpy as np
import os
import time
from catboost import CatBoostRegressor
from xgboost import XGBRegressor 
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import mean_squared_log_error, mean_squared_error
from lightgbm import LGBMRegressor
import optuna

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
submission = pd.read_csv('data/sample_submission.csv')

In [ ]:
def add_feature_cross_terms(df, features):
    df_new = df.copy()
    for i in range(len(features)):
        for j in range(i + 1, len(features)):
            f1 = features[i]
            f2 = features[j]
            df_new[f"{f1}_x_{f2}"] = df_new[f1] * df_new[f2]
    return df_new

num_features = ['Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp']
train = add_feature_cross_terms(train, num_features)
test = add_feature_cross_terms(test, num_features)

In [ ]:
train['Sex'] = train['Sex'].map({'male' : 0, 'female' : 1}).astype('category')
test['Sex'] = test['Sex'].map({'male' : 0, 'female' : 1}).astype('category')

In [ ]:
X = train.drop(columns=['id', 'Calories'])
y = np.log1p(train['Calories'])
X_test = test.drop(columns=['id'])

In [ ]:
oof_cb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))
oof_lgbm = np.zeros(len(train))
pred_cb = np.zeros(len(test))
pred_xgb = np.zeros(len(test))
pred_lgbm = np.zeros(len(test))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
from catboost import CatBoostRegressor

def objective(trial):
    params = {
        "iterations": 3000,
        "learning_rate": 0.02,
        "depth": trial.suggest_int("depth", 10, 16),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.001, 10.0, log=True),
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "random_seed": 42
    }
    # {'iterations': 3083, 'learning_rate': 0.020331009302072385, 'depth': 14, 'l2_leaf_reg': 0.670045854544982}. Best is trial 6 with value: 0.0592121306427973.
    # Split the data into a training and validation set
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = CatBoostRegressor(
        **params,
        cat_features=[X.columns.get_loc('Sex')],
        verbose=0
    )
    
    # Train the model with early stopping on the validation set
    model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100)
    
    # Get predictions on the validation set and compute RMSE
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    
    trial.set_user_attr("rmse", rmse)
    
    return rmse

# Example usage with Optuna:
# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=100)

In [ ]:
study = optuna.create_study(direction="minimize")

study.optimize(objective, n_trials=50)

print("Best trial:")
trial = study.best_trial
print("  RMSE: {}".format(trial.value))
print("  Best hyperparameters: {}".format(trial.params))